# LifeMate — ASR dataset audit & model selection

Notebook riêng tư cho đội đã đăng ký. **Không publish notebook có output, audio, transcript hoặc CSV của cuộc thi.**

Input mặc định: `/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset`.

**Mục tiêu:** kiểm kê schema; liên kết audio–transcript ở yêu cầu cuối và history user; khảo sát định dạng, độ dài, chất lượng tín hiệu, văn bản và nhóm lỗi quan trọng; tạo validation có kiểm tra leakage; tùy chọn chạy baseline và so sánh công bằng.

**Giới hạn:** tác giả notebook chưa truy cập được mount Kaggle này. Adapter tự dò schema có báo cáo coverage, nhưng phải được kiểm tra trên dữ liệu thật. Notebook không thể xác định “model tốt nhất” từ EDA; cần kết quả benchmark và quy định cuộc thi. Không tự suy ra accent, giới tính, tuổi hoặc SNR từ waveform.

**Ranh giới:** không đọc, liệt kê nội dung hoặc giải nén thư mục `*_environment*`, `environment` hay bất kỳ `state.json`; không dùng label tool calls/final state làm đầu vào ASR. Không gọi API nhận dạng bên ngoài. Chạy model local trên Kaggle chỉ khi quy định cho phép.

## Cách chạy
1. Add Input dataset; giữ notebook **Private**. Chạy phần A–F trên CPU trước.
2. Kiểm tra `schema_paths`, `resolution_status`, coverage audio/text; sửa cấu hình nếu schema khác.
3. Nghe bảng review, bổ sung nhãn thủ công; xem split và các giới hạn trước khi benchmark.
4. Phần G mặc định **tắt**. Muốn chạy: bật GPU, chuẩn bị checkpoint/Internet theo quy định, bật `RUN_BASELINES=True`, chạy lại từ đầu.
5. File riêng tư xuất vào `/kaggle/working/lifemate_asr_audit`. Chỉ chia sẻ trong phạm vi cuộc thi cho phép.


## A. Cấu hình và phụ thuộc
EDA dùng NumPy, Pandas, SciPy, Matplotlib, SoundFile có sẵn phổ biến trên Kaggle. Cell dưới không tự cài hoặc nâng cấp package. Nếu thiếu, cài ở cell riêng khi Internet/quy định cho phép: `%pip install soundfile scipy pandas matplotlib psutil`.

Benchmark tùy chọn dùng Transformers/PyTorch. Không cần GPU cho phần khảo sát. Mọi ngưỡng chất lượng bên dưới là **ngưỡng sàng lọc ban đầu**, không phải tiêu chuẩn cuộc thi.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os, re, json, math, time, random, hashlib, unicodedata, difflib, gc, platform
import importlib.metadata as im
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown

ROOT = Path('/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset')
OUT = Path('/kaggle/working/lifemate_asr_audit')
SEED = 2026
MAX_SIGNAL_FILES = 2000  # None: audit tín hiệu mọi file; mặc định lấy mẫu theo split/độ dài
MAX_SIGNAL_SECONDS = 120  # file dài hơn: giữ header, đánh dấu chưa audit tín hiệu toàn file
RUN_FILE_HASH = True  # SHA256 byte file: chậm hơn nhưng phát hiện file copy y hệt
RUN_BASELINES = False
BASELINE_N = 100  # subset cố định của validation; không tự gọi đây là toàn bộ validation
VAL_FRACTION = 0.20
USE_SPEAKER_GROUPS = True  # chỉ khi có speaker_id tường minh; không dùng user profile để đoán
MANIFESTS = []  # để trống: tìm *.jsonl, train.json/test.json/public_test.json/samples.json
SPLIT_OVERRIDES = {}  # {'folder/custom.jsonl': 'train'}; tên không rõ sẽ là unknown
AUDIO_KEYS = {'audio','audio_path','audio_file','audio_filepath','audio_url','wav','wav_path','file_path','path'}
TEXT_KEYS = ('audio_text', 'transcript')  # KHÔNG mặc định dùng mọi trường `text`
ALLOW_HISTORY_TEXT = False  # chỉ bật sau khi xác minh history.text là transcript chuẩn
PATH_OVERRIDES = {}  # {'audio_ref_trong_json': 'train/audio/abc.wav'}
AUDIO_EXTS = {'.wav','.flac','.mp3','.ogg','.opus','.m4a','.aac','.wma','.aiff','.aif'}
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
rng = np.random.default_rng(SEED)
assert ROOT.is_dir(), f'Không thấy dataset: {ROOT}. Kiểm tra Add Input và sửa ROOT.'

def forbidden(p):
    p = Path(p)
    return p.name.lower() == 'state.json' or any(
        '_environment' in x.lower() or x.lower() == 'environment' for x in p.parts)

def safe(p):
    p = Path(p)
    if forbidden(p):
        raise PermissionError('Đường dẫn environment/state bị chặn')
    q = p.resolve()
    if forbidden(q) or not q.is_relative_to(ROOT.resolve()):
        raise PermissionError('Đường dẫn ngoài ROOT hoặc qua symlink bị chặn')
    return q

def save_csv(df, name):
    df.to_csv(OUT / name, index=False, encoding='utf-8-sig')

versions = {'python': platform.python_version()}
for package in ['numpy','pandas','scipy','soundfile','torch','transformers','psutil']:
    try: versions[package] = im.version(package)
    except im.PackageNotFoundError: versions[package] = 'not installed'
display(pd.Series(versions, name='version').to_frame())


## B. Kiểm kê an toàn và hiểu schema
Không mở JSON environment. `os.walk` được prune trước khi đi vào các thư mục bị chặn; symlink không được theo. Chỉ xem **tên trường/kiểu**, không in toàn bộ sample hoặc profile. Nếu dữ liệu chỉ có ZIP, notebook dừng và yêu cầu chuẩn bị input đã giải nén đúng phạm vi; không tự giải nén tất cả.


In [ ]:
files = []
for base, dirs, names in os.walk(safe(ROOT), followlinks=False):
    dirs[:] = sorted(d for d in dirs if not forbidden(Path(base)/d) and not (Path(base)/d).is_symlink())
    for name in sorted(names):
        p = Path(base)/name
        if forbidden(p) or p.is_symlink(): continue
        files.append(safe(p))
inventory = pd.DataFrame([{'relative_path': str(p.relative_to(ROOT.resolve())),
                           'extension': p.suffix.lower(), 'bytes': p.stat().st_size} for p in files])
if inventory.empty: raise RuntimeError('Không có file được phép đọc trong ROOT.')
display(inventory.groupby('extension').agg(files=('bytes','size'), MB=('bytes', lambda s:s.sum()/1e6)))
save_csv(inventory, 'inventory.csv')
audio_files = [p for p in files if p.suffix.lower() in AUDIO_EXTS]
if MANIFESTS:
    manifest_paths = [safe(ROOT/x) for x in MANIFESTS]
else:
    manifest_paths = [p for p in files if p.suffix.lower()=='.jsonl' or p.name.lower() in
                      {'train.json','test.json','public_test.json','samples.json'}]
assert manifest_paths, 'Không tìm thấy manifest. Nếu chỉ có ZIP, chuẩn bị input đã giải nén an toàn; hoặc đặt MANIFESTS.'
assert audio_files, 'Không thấy file audio. Kiểm tra thư mục lồng/định dạng/archives.'

def infer_split(p):
    rel = str(p.relative_to(ROOT.resolve()))
    if rel in SPLIT_OVERRIDES: return SPLIT_OVERRIDES[rel]
    words = set(re.split(r'[^a-z0-9]+', rel.lower()))
    if 'train' in words: return 'train'
    if words & {'test','publictest','public'}: return 'test'
    if words & {'val','valid','validation','dev'}: return 'provided_validation'
    return 'unknown'

def read_records(p):
    p = safe(p)
    if p.suffix.lower()=='.jsonl':
        with p.open(encoding='utf-8-sig') as f:
            for line_no, line in enumerate(f, 1):
                if not line.strip(): continue
                try:
                    item = json.loads(line)
                    if not isinstance(item, dict): raise ValueError('Sample phải là object')
                    yield line_no, item
                except Exception as e:
                    parse_errors.append({'manifest':str(p), 'line':line_no,'error':str(e)[:200]})
    else:
        obj = json.loads(p.read_text(encoding='utf-8-sig'))
        if isinstance(obj, dict):
            obj = next((obj[k] for k in ['samples','data','records'] if isinstance(obj.get(k), list)), [obj])
        if not isinstance(obj, list): raise ValueError(f'JSON format chưa hỗ trợ: {p.name}')
        for i, item in enumerate(obj, 1):
            if isinstance(item, dict): yield i, item
            else: parse_errors.append({'manifest':str(p),'line':i,'error':'non-object record'})

schema = Counter()
def inspect_schema(obj, prefix='$', depth=0):
    if depth>7: return
    schema[(prefix, type(obj).__name__)] += 1
    if isinstance(obj, dict):
        for k,v in obj.items():
            if k in {'label','tool_calls','tool_results','final_state'}: continue
            inspect_schema(v, prefix+'.'+k, depth+1)
    elif isinstance(obj,list):
        for v in obj[:3]: inspect_schema(v, prefix+'[]', depth+1)

parse_errors, samples = [], []
for manifest in manifest_paths:
    for line_no, obj in read_records(manifest):
        if len(samples)<200: inspect_schema(obj)
        uid = f'{manifest.relative_to(ROOT.resolve())}:{line_no}'
        samples.append({'sample_uid':uid,'sample_id':str(obj.get('id',obj.get('sample_id',line_no))),
                        'split':infer_split(manifest),'manifest':str(manifest),'obj':obj})
schema_df = pd.DataFrame([{'path':p,'type':t,'observations':n} for (p,t),n in schema.items()])
display(schema_df.head(100))
save_csv(schema_df,'schema_paths.csv')
save_csv(pd.DataFrame(parse_errors,columns=['manifest','line','error']),'parse_errors.csv')
print('Manifest:',len(manifest_paths),'| samples:',len(samples),'| audio files:',len(audio_files), '| parse errors:',len(parse_errors))
display(pd.Series([s['split'] for s in samples]).value_counts().rename('samples'))


### Liên kết yêu cầu cuối và history
Adapter chỉ tìm audio trong sample chính và lượt history user/human; bỏ context, label và tool fields. Với role thiếu, giữ nhãn `unknown` để review, không âm thầm coi là user. Transcript lấy `audio_text`/`transcript` cùng object hoặc kế thừa trong cùng lượt; không dùng transcript yêu cầu cuối cho history. Mọi audio ref được lưu kèm đường dẫn trường để kiểm tra.

Resolver chỉ nhận file có trong inventory an toàn: ưu tiên đường dẫn chính xác, sau đó suffix duy nhất; basename chỉ dùng nếu duy nhất toàn dataset. Trường hợp mơ hồ không tự chọn file. Audio không được manifest tham chiếu vẫn được audit header, nhưng không có đáp án để benchmark.


In [ ]:
SKIP_FIELDS = {'context','label','tool_calls','tool_results','final_state','tools','tools_catalog','history'}
by_basename = defaultdict(list)
suffix_map = defaultdict(list)
for p in audio_files:
    by_basename[p.name].append(p)
    parts = p.relative_to(ROOT.resolve()).parts
    for n in range(2,len(parts)+1): suffix_map['/'.join(parts[-n:])].append(p)
audio_set = set(audio_files)

def resolve_ref(ref, manifest):
    ref = PATH_OVERRIDES.get(ref,ref).replace('\\','/').strip()
    if forbidden(Path(ref)): return None,'blocked'
    if '://' in ref: return None,'remote_not_fetched'
    for p in [Path(ref), Path(manifest).parent/ref, ROOT/ref]:
        try:
            q = safe(p)
            if q in audio_set: return str(q),'exact'
        except (PermissionError,OSError): pass
    parts = [x for x in ref.split('/') if x not in {'','.','..'}]
    for start in range(max(0,len(parts)-1)):
        hits = suffix_map.get('/'.join(parts[start:]),[])
        if len(hits)==1: return str(hits[0]),'unique_suffix'
        if len(hits)>1: return None,'ambiguous_suffix'
    hits = by_basename.get(parts[-1],[]) if parts else []
    if len(hits)==1: return str(hits[0]),'unique_basename_review'
    return None,'ambiguous_basename' if hits else 'missing'

def find_audio(unit, prefix='$', inherited_text=None, inherited_speaker=None):
    if isinstance(unit, dict):
        text = next((unit[k] for k in TEXT_KEYS if isinstance(unit.get(k),str)), inherited_text)
        speaker = unit.get('speaker_id', inherited_speaker)
        for k,v in unit.items():
            if k in SKIP_FIELDS: continue
            field = prefix+'.'+k
            if isinstance(v,str) and k.lower() in AUDIO_KEYS and (
                k.lower() not in {'path','file_path'} or Path(v).suffix.lower() in AUDIO_EXTS):
                yield field,v,text,speaker
            elif isinstance(v,(dict,list)):
                yield from find_audio(v,field,text,speaker)
    elif isinstance(unit,list):
        for i,v in enumerate(unit):
            if isinstance(v,str) and Path(v).suffix.lower() in AUDIO_EXTS:
                yield f'{prefix}[{i}]',v,inherited_text,inherited_speaker
            elif isinstance(v,(dict,list)):
                yield from find_audio(v,f'{prefix}[{i}]',inherited_text,inherited_speaker)

rows, sample_stats = [], []
for s in samples:
    obj = s['obj']
    hist = obj.get('history',[])
    if isinstance(hist,dict): hist = hist.get('messages',hist.get('turns',[]))
    if not isinstance(hist,list): hist=[]
    units = [('final',-1,'user',obj)]
    for i,turn in enumerate(hist):
        if not isinstance(turn,dict): continue
        role = str(turn.get('role',turn.get('speaker','unknown'))).lower()
        if role in {'user','human','unknown'}:
            turn = dict(turn)
            if ALLOW_HISTORY_TEXT and not any(k in turn for k in TEXT_KEYS) and isinstance(turn.get('text'),str):
                turn['transcript'] = turn['text']
            units.append(('history',i,role,turn))
    count_final=count_hist=0
    for kind,idx,role,unit in units:
        for field,ref,text,speaker in find_audio(unit):
            path,status = resolve_ref(ref,s['manifest'])
            count_final += kind=='final'
            count_hist += kind=='history'
            rows.append({k:s[k] for k in ['sample_uid','sample_id','split','manifest']} |
                        {'kind':kind,'turn_index':idx,'role':role,'field_path':field,
                         'audio_ref':ref,'path':path,'resolution_status':status,'reference':text,
                         'speaker_id':str(speaker) if speaker is not None else None})
    sample_stats.append({k:s[k] for k in ['sample_uid','split']} |
                        {'history_turns':len(hist),'final_audio_refs':count_final,'history_audio_refs':count_hist})
refs = pd.DataFrame(rows)
assert not refs.empty, 'Adapter chưa thấy audio ref. Xem schema_paths, sửa AUDIO_KEYS/adapter; KHÔNG đoán cặp file–text.'
refs['has_reference'] = refs['reference'].fillna('').str.strip().ne('')
refs['pairing_status']='ok'
# Không chấm từng clip bằng transcript ghép cho nhiều clip khi schema chưa xác minh.
for _,g in refs[refs.has_reference].groupby(['sample_uid','kind','turn_index','reference'],dropna=False):
    if g.path.nunique()>1:
        refs.loc[g.index,'pairing_status']='shared_text_multiple_audio_review'
# Hai field alias cùng trỏ một file trong cùng lượt không tạo hai quan sát độc lập.
refs=refs.drop_duplicates(['sample_uid','kind','turn_index','path','audio_ref','reference']).reset_index(drop=True)
refs['ref_uid'] = [f'r{i:08d}' for i in range(len(refs))]
stats = pd.DataFrame(sample_stats)
display(refs.groupby(['split','kind','resolution_status'],dropna=False).size().rename('references').to_frame())
display(refs.groupby(['split','kind']).agg(refs=('ref_uid','size'),with_text=('has_reference','sum')))
display(refs.pairing_status.value_counts().rename('pairing_status'))
display(stats.groupby('split')[['history_turns','final_audio_refs','history_audio_refs']].describe())
print('Samples không tìm thấy final audio ref:',int((stats.final_audio_refs==0).sum()))
print('Audio paths không được tham chiếu:',len(audio_set-set(Path(p) for p in refs.path.dropna())))
save_csv(refs,'audio_references.csv'); save_csv(stats,'sample_coverage.csv')
assert refs.path.notna().any(), 'Không resolve được audio nào. Kiểm tra ROOT/PATH_OVERRIDES/schema.'


## C. Audio: header toàn bộ, tín hiệu lấy mẫu có kiểm soát
Header: duration, sample rate, channels, subtype/bit depth nếu định dạng hỗ trợ, codec/container, đọc lỗi. Không suy bit depth gốc từ dtype float đã decode.

Tín hiệu: peak, RMS dBFS (mức tín hiệu số), DC offset, zero fraction, tỷ lệ gần full-scale, chênh lệch kênh, tỷ lệ frame yên lặng theo ngưỡng năng lượng. **Không gọi các chỉ số này là SNR hoặc VAD chuẩn.** Gain/compression làm thay đổi chúng; clipping lossy cũng có thể không chạm ±1.0.

File dài hơn `MAX_SIGNAL_SECONDS` chỉ audit header, không báo chất lượng toàn file từ đoạn đầu. Số file tín hiệu thực sự kiểm tra được luôn được báo riêng.


In [ ]:
def file_sha256(p):
    h=hashlib.sha256()
    with safe(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

headers=[]
for i,p in enumerate(audio_files):
    row={'path':str(p),'file_bytes':p.stat().st_size,'extension':p.suffix.lower()}
    try:
        inf=sf.info(str(safe(p)))
        row.update(sr=inf.samplerate,channels=inf.channels,frames=inf.frames,
                   duration=inf.duration,format=inf.format,subtype=inf.subtype,header_error=None)
    except Exception as e: row['header_error']=f'{type(e).__name__}: {e}'[:250]
    if RUN_FILE_HASH:
        try: row['file_sha256']=file_sha256(p)
        except Exception as e: row['hash_error']=str(e)[:160]
    headers.append(row)
    if (i+1)%1000==0: print('Headers:',i+1,'/',len(audio_files))
audio_df=pd.DataFrame(headers)
for c in ['sr','channels','frames','duration','format','subtype','file_sha256']:
    if c not in audio_df: audio_df[c]=np.nan
split_per_path=refs.dropna(subset=['path']).groupby('path')['split'].agg(lambda s:'+'.join(sorted(set(s))))
audio_df['ref_split']=audio_df.path.map(split_per_path).fillna('unreferenced')
display(audio_df.groupby(['format','subtype','sr','channels'],dropna=False).size().rename('files').to_frame())
display(audio_df.groupby('ref_split').duration.describe(percentiles=[.5,.9,.95,.99]))
display(audio_df[audio_df.header_error.notna()][['path','header_error']].head(10))
print('Tổng giờ audio decode được:',audio_df.duration.sum()/3600)
print('Header failures:',audio_df.header_error.notna().sum(),'/',len(audio_df))
save_csv(audio_df,'audio_headers.csv')
good=audio_df[audio_df.header_error.isna() & audio_df.duration.gt(0)].copy()
assert len(good), 'SoundFile không đọc được audio nào. Kiểm tra codec; cần backend hỗ trợ, không bỏ qua lỗi này.'
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].hist(good.duration,bins=40); axs[0].set(xlabel='Duration (s)',ylabel='Files',title='All readable files')
axs[1].hist(good.duration.clip(upper=good.duration.quantile(.99)),bins=40)
axs[1].set(xlabel='Duration clipped at P99 for visualization (s)',title='Chi tiet phan lon du lieu')
plt.tight_layout(); plt.savefig(OUT/'duration.png',dpi=140); plt.show()


In [ ]:
good['duration_bin']=pd.cut(good.duration,[0,3,10,30,60,120,np.inf],include_lowest=True).astype(str)
# Round-robin lấy mẫu theo split × độ dài, tái lập được, không chỉ đọc N file đầu.
buckets=[list(g.sample(frac=1,random_state=SEED).index)
         for _,g in good.groupby(['ref_split','duration_bin'],observed=True)]
selected=[]
while any(buckets) and (MAX_SIGNAL_FILES is None or len(selected)<MAX_SIGNAL_FILES):
    for b in buckets:
        if b and (MAX_SIGNAL_FILES is None or len(selected)<MAX_SIGNAL_FILES): selected.append(b.pop())
signal_rows=[]
for ix in selected:
    a=good.loc[ix]; r={'path':a.path,'signal_status':'ok'}
    if a.duration>MAX_SIGNAL_SECONDS:
        r['signal_status']='skipped_too_long'; signal_rows.append(r); continue
    try:
        x,sr=sf.read(str(safe(a.path)),dtype='float32',always_2d=True)
        if not len(x) or not np.isfinite(x).all(): raise ValueError('empty/nonfinite samples')
        ch_rms=np.sqrt(np.mean(x.astype('float64')**2,axis=0))
        mono=x.mean(axis=1)
        rms=float(np.sqrt(np.mean(mono.astype('float64')**2)))
        win=max(1,round(.025*sr)); hop=max(1,round(.010*sr))
        if len(mono)>=win:
            cs=np.r_[0,np.cumsum(mono.astype('float64')**2)]
            starts=np.arange(0,len(mono)-win+1,hop)
            frame_db=10*np.log10((cs[starts+win]-cs[starts])/win+1e-12)
        else: frame_db=np.array([20*np.log10(rms+1e-12)])
        threshold=max(-55.,float(frame_db.max())-40.)
        # Hash decode float32 cùng sample rate và channel; không chuẩn hóa gain/resample.
        h=hashlib.sha256(f'{sr}:{x.shape[1]}:'.encode()+x.astype('<f4').tobytes()).hexdigest()
        r.update(peak=float(np.max(np.abs(x))),rms_dbfs=20*np.log10(rms+1e-12),
                 dc_offset=float(mono.mean()),near_fullscale_fraction=float((np.abs(x)>=.999).mean()),
                 zero_fraction=float((x==0).mean()),quiet_frame_fraction=float((frame_db<threshold).mean()),
                 channel_rms_ratio=float(ch_rms.max()/(ch_rms.min()+1e-12)),
                 mono_to_loudest_rms=float(rms/(ch_rms.max()+1e-12)),decoded_sha256=h)
    except Exception as e: r.update(signal_status='decode_error',signal_error=str(e)[:200])
    signal_rows.append(r)
signal_df=pd.DataFrame(signal_rows)
for c in ['peak','rms_dbfs','dc_offset','near_fullscale_fraction','zero_fraction','quiet_frame_fraction',
          'channel_rms_ratio','mono_to_loudest_rms','decoded_sha256']:
    if c not in signal_df: signal_df[c]=np.nan
audio_df=audio_df.merge(signal_df,on='path',how='left')
audio_df['signal_status']=audio_df.signal_status.fillna('not_selected')
audio_df['flag_near_fullscale']=audio_df.near_fullscale_fraction.gt(.001)
audio_df['flag_low_level']=audio_df.rms_dbfs.lt(-40)
audio_df['flag_many_quiet_frames']=audio_df.quiet_frame_fraction.gt(.50)
audio_df['flag_channel_cancellation']=audio_df.channels.gt(1)&audio_df.mono_to_loudest_rms.lt(.25)
display(audio_df.signal_status.value_counts().rename('files'))
display(audio_df.groupby('ref_split').agg(files=('path','size'),signal_ok=('signal_status',lambda s:(s=='ok').sum())))
display(audio_df[['peak','rms_dbfs','near_fullscale_fraction','quiet_frame_fraction','mono_to_loudest_rms']].describe())
save_csv(audio_df,'audio_audit.csv')


## D. Transcript, nội dung quan trọng và khác biệt train/test
Phân tích văn bản chỉ dùng transcript đã cung cấp, không dùng tool label. WER whitespace tiếng Việt thường gần mức âm tiết, không tự gọi là WER từ vựng đã word-segment. Giữ dấu tiếng Việt và từ phủ định. Tạo bản normalization nhẹ để so sánh cách viết; **không đổi số bằng chữ thành chữ số** khi chưa có quy tắc được xác minh.

Các regex sau là **cờ để chọn mẫu review**, không phải bộ trích xuất entity chính xác. Latin không dấu không đủ để kết luận tiếng Anh/code-switching; viết hoa không đủ để kết luận tên người. Không suy ra phân bố vùng miền từ các từ trong transcript.


In [ ]:
def normalize_text(text, punctuation=True):
    text=unicodedata.normalize('NFC',str(text or '')).casefold()
    if punctuation:
        text=''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text)
    return ' '.join(text.split())

patterns={
 'money_hint':r'\b(?:nghìn|ngàn|triệu|tỷ|đồng|vnd|vnđ|usd)\b|[₫$]',
 'time_hint':r'\b(?:hôm nay|ngày mai|sáng mai|tối mai|giờ|phút|thứ hai|tháng|tuần)\b|\b\d{1,2}:\d{2}\b',
 'negation_hint':r'\b(?:không|đừng|chưa|chẳng|chớ)\b',
 'number_word_hint':r'\b(?:một|hai|ba|bốn|năm|sáu|bảy|tám|chín|mười|trăm|nghìn|ngàn|triệu)\b',
 'location_hint':r'\b(?:đường|quận|huyện|phường|tỉnh|thành phố|địa chỉ)\b',
 'device_app_hint':r'\b(?:đèn|máy lạnh|điều hòa|tivi|loa|điện thoại|calendar|gmail|zalo|spotify)\b',
 'english_hint':r'\b(?:google|calendar|meeting|email|gmail|youtube|spotify|wifi|bluetooth|deadline|reminder)\b',
 'person_hint':r'\b(?:anh|chị|cô|chú|bác|ông|bà|bạn|em)\s+[^\W\d_]+',
}
refs['reference_norm']=refs.reference.fillna('').map(normalize_text)
refs['n_space_tokens']=refs.reference_norm.str.split().str.len()
refs['n_chars']=refs.reference_norm.str.len()
refs['has_digits']=refs.reference.fillna('').str.contains(r'\d',regex=True)
refs['has_punctuation']=refs.reference.fillna('').map(lambda s:any(unicodedata.category(c).startswith('P') for c in s))
refs['has_diacritics']=refs.reference.fillna('').map(lambda s:any(unicodedata.combining(c) for c in unicodedata.normalize('NFD',s)))
for name,pat in patterns.items(): refs[name]=refs.reference_norm.str.contains(pat,regex=True)
meta_cols=['path','duration','sr','channels','rms_dbfs','quiet_frame_fraction','signal_status',
           'file_sha256','decoded_sha256','flag_near_fullscale','flag_low_level','flag_many_quiet_frames','flag_channel_cancellation']
refs=refs.merge(audio_df[meta_cols],on='path',how='left')
refs['tokens_per_audio_second']=refs.n_space_tokens/refs.duration.replace(0,np.nan)
refs['reference_status']=np.where(refs.has_reference,'provided','missing')
text_rows=refs[refs.has_reference]
display(text_rows.groupby(['split','kind'])[['n_space_tokens','n_chars','tokens_per_audio_second']].describe())
display(text_rows.groupby(['split','kind'])[list(patterns)+['has_digits','has_diacritics','has_punctuation']].mean().round(3))
print('Text giống hệt sau normalization:',text_rows.reference_norm.duplicated().sum())
print('Audio có nhiều reference khác nhau:',refs.dropna(subset=['path']).query('has_reference').groupby('path').reference_norm.nunique().gt(1).sum())
print('Public test không có text -> không suy phân bố entity/accent/code-switching từ nhãn không tồn tại.')
display(refs.groupby(['split','kind']).agg(refs=('ref_uid','size'),resolved=('path','count'),
      texts=('has_reference','sum'),duration_median=('duration','median'),duration_p95=('duration',lambda s:s.quantile(.95))))
save_csv(refs,'references_enriched.csv')


### Khối lượng inference và vocabulary
ASR phải xử lý final audio và có thể cả history user. Tính tổng thời lượng **các file duy nhất trong mỗi sample**, không cộng alias một file nhiều lần. History có thể lặp giữa các sample: cache chỉ là cơ hội tối ưu, cần xác minh cách cấp audio lúc thi và không cache transcript chuẩn vào inference. Tốc độ token/giây dưới đây bao gồm khoảng im lặng; không phải tốc độ phát âm đã căn chỉnh.


In [ ]:
work=refs.dropna(subset=['path']).drop_duplicates(['sample_uid','kind','path'])
workload=work.groupby(['sample_uid','split','kind']).duration.sum().unstack('kind',fill_value=0).reset_index()
for k in ['final','history']:
    if k not in workload: workload[k]=0.
workload['total_audio_seconds']=workload['final']+workload['history']
display(workload.groupby('split')[['final','history','total_audio_seconds']].describe(percentiles=[.5,.9,.95]))
print('References / unique resolved paths:',refs.path.notna().sum(),'/',refs.path.nunique())
print('Float32 mono 16kHz RAM cho waveform 1 phút ~',16000*60*4/2**20,'MiB, chưa gồm features/model/activation.')
train_text=refs[(refs.split=='train')&refs.has_reference&refs.pairing_status.eq('ok')].drop_duplicates(['sample_uid','kind','turn_index','reference_norm'])
vocab=Counter(t for text in train_text.reference_norm for t in text.split())
vocab_df=pd.DataFrame(vocab.most_common(),columns=['space_token','count'])
save_csv(vocab_df,'train_token_frequencies.csv'); save_csv(workload,'sample_audio_workload.csv')
print('Whitespace vocabulary:',len(vocab),'| singleton tokens:',sum(n==1 for n in vocab.values()))
print('Không coi token hiếm là tên riêng nếu chưa review; không in toàn bộ vocabulary mặc định.')


## E. Leakage và validation có thể tái lập
Train/validation phải tách theo **sample hội thoại**, không tách final/history của cùng sample. Nhóm thêm audio byte/decoded trùng, transcript chuẩn hóa trùng và speaker_id tường minh nếu có. Transcript ngắn lặp phổ biến có thể tạo nhóm lớn; notebook báo kích thước nhóm thay vì âm thầm phá nhóm.

Near-duplicate text dưới đây chỉ là heuristic dùng blocking theo độ dài + từ đầu/cuối, giới hạn số cặp. Đây không phải đảm bảo loại bỏ mọi câu gần giống, cũng không phát hiện audio cắt lại/chỉnh gain/tốc độ. Nếu không có speaker metadata, **không thể khẳng định speaker-disjoint**.

Public test chỉ được kiểm tra overlap kỹ thuật; không dùng transcript test hoặc leaderboard để chọn cấu hình. Nếu phát hiện overlap với test, loại sample train liên quan khỏi tập benchmark này để tránh đánh giá dễ giả tạo.


In [ ]:
train=refs[(refs.split=='train') & refs.role.isin(['user','human']) & refs.has_reference & refs.path.notna()
           & refs.pairing_status.eq('ok')].copy()
assert len(train), 'Không có train audio-text hợp lệ. Sửa SPLIT_OVERRIDES/adapter; không tự coi test là train.'
parent={s:s for s in train.sample_uid.unique()}
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[b]=a
def unite_by(frame,column):
    for _,g in frame.dropna(subset=[column]).groupby(column):
        ids=g.sample_uid.unique()
        for uid in ids[1:]: union(ids[0],uid)

for col in ['path','file_sha256','decoded_sha256','reference_norm']:
    unite_by(train[train[col].astype(str).ne('')],col)
speaker_coverage=float(train.speaker_id.notna().mean())
if USE_SPEAKER_GROUPS: unite_by(train,'speaker_id')

MAX_NEAR_PAIRS=100000
blocks=defaultdict(list)
unique_text=train.drop_duplicates('reference_norm')
for _,r in unique_text.iterrows():
    toks=r.reference_norm.split()
    if len(toks)<5: continue
    # Dùng hai độ dài lân cận để bớt bỏ sót câu thêm/bớt một token.
    for size in {len(toks)//3,(len(toks)+1)//3}:
        blocks[(size,toks[0],toks[-1])].append((r.sample_uid,r.reference_norm))
near_rows=[]; seen_pairs=set(); pair_count=0; truncated=False
for block in blocks.values():
    if truncated: break
    for i,(a,ta) in enumerate(block):
        if truncated: break
        for b,tb in block[i+1:]:
            key=tuple(sorted((a,b)))
            if a==b or key in seen_pairs: continue
            if pair_count>=MAX_NEAR_PAIRS: truncated=True; break
            seen_pairs.add(key); pair_count+=1
            score=difflib.SequenceMatcher(None,ta,tb,autojunk=False).ratio()
            if score>=.92:
                union(a,b); near_rows.append({'sample_a':a,'sample_b':b,'text_similarity':score})
train['group_id']=train.sample_uid.map(find)

overlaps=[]; contaminated=set()
test=refs[refs.split=='test']
for col in ['path','file_sha256','decoded_sha256']:
    keys=set(test[col].dropna())-set([''])
    hits=train[train[col].isin(keys)]
    contaminated.update(hits.sample_uid)
    for _,r in hits.iterrows(): overlaps.append({'sample_uid':r.sample_uid,'ref_uid':r.ref_uid,'overlap_type':col})
blocked_groups=set(train[train.sample_uid.isin(contaminated)].group_id)
eligible=train[~train.group_id.isin(blocked_groups)].copy()
groups=eligible.groupby('group_id').sample_uid.nunique().sort_values(ascending=False)
display(groups.head(10).rename('samples_in_group').to_frame())
print('Speaker metadata coverage:',speaker_coverage,'| near pairs checked:',pair_count,'| truncated:',truncated)
print('Train/test technical overlaps:',len(overlaps),'| blocked groups:',len(blocked_groups))
group_ids=groups.index.to_numpy().copy(); rng.shuffle(group_ids)
target=max(1,round(eligible.sample_uid.nunique()*VAL_FRACTION))
val_groups=set(); n=0
if len(group_ids)>=2:
    for g in group_ids[:-1]:
        if n>=target: break
        val_groups.add(g); n+=int(groups[g])
eligible['partition']=np.where(eligible.group_id.isin(val_groups),'validation','development')
if not val_groups:
    print('CHƯA TẠO VALIDATION: dữ liệu chỉ có <=1 nhóm độc lập. Cần xem lại nhóm/metadata, không fallback random.')
split_manifest=eligible[['ref_uid','sample_uid','group_id','partition','kind','path','reference']].copy()
validation=eligible[eligible.partition=='validation'].copy()
development=eligible[eligible.partition=='development'].copy()
for col in ['sample_uid','group_id','path','file_sha256','decoded_sha256','reference_norm']:
    assert not (set(validation[col].dropna()) & set(development[col].dropna())), f'Leakage: {col}'
if USE_SPEAKER_GROUPS:
    assert not (set(validation.speaker_id.dropna()) & set(development.speaker_id.dropna()))
display(eligible.groupby(['partition','kind']).agg(refs=('ref_uid','size'),samples=('sample_uid','nunique'),hours=('duration',lambda s:s.sum()/3600)))
print('Speaker-disjoint chỉ xác minh trên ID đã cung cấp; độ phủ thiếu -> chưa chứng minh cho toàn tập.')
save_csv(split_manifest,'split_manifest.csv')
save_csv(pd.DataFrame(overlaps,columns=['sample_uid','ref_uid','overlap_type']),'train_test_overlap.csv')
save_csv(pd.DataFrame(near_rows,columns=['sample_a','sample_b','text_similarity']),'near_text_pairs.csv')


## F. Nghe kiểm tra và annotation thủ công
Xuất hàng đợi ưu tiên: audio dài/ngắn, mức tín hiệu thấp, kênh có nguy cơ triệt tiêu, tên/số/thời gian/phủ định, history và mẫu ngẫu nhiên. Không bật audio hàng loạt, không in profile. Nhãn giọng/noise/code-switching cần **nghe và điền**, có lựa chọn `uncertain`.

Điền `manual_annotations.csv`, sau đó có thể đọc lại bằng `pd.read_csv(...)`. Tỷ lệ từ queue ưu tiên **không phải ước lượng tỷ lệ toàn dataset**. Muốn ước lượng phân bố, dùng riêng mẫu random và ghi cỡ mẫu/độ bất định.


In [ ]:
pool=refs[refs.path.notna() & refs.duration.notna()].copy()
queue_parts=[pool.sample(n=min(40,len(pool)),random_state=SEED).assign(selection_reason='random')]
for name in list(patterns)+['flag_low_level','flag_near_fullscale','flag_many_quiet_frames','flag_channel_cancellation']:
    g=pool[pool[name].fillna(False)]
    if len(g): queue_parts.append(g.sample(n=min(8,len(g)),random_state=SEED).assign(selection_reason=name))
queue_parts += [pool.nlargest(8,'duration').assign(selection_reason='longest'),
                pool.nsmallest(8,'duration').assign(selection_reason='shortest')]
review=pd.concat(queue_parts).drop_duplicates('ref_uid').copy()
for c in ['accent_manual','noise_manual','code_switch_manual','speaker_count_manual',
          'reference_problem_manual','channel_choice_manual','notes']:
    review[c]=''
review_cols=['ref_uid','path','split','kind','reference','duration','selection_reason',
             'accent_manual','noise_manual','code_switch_manual','speaker_count_manual',
             'reference_problem_manual','channel_choice_manual','notes']
save_csv(review[review_cols],'manual_annotations.csv')
display(review[['ref_uid','split','kind','duration','selection_reason']].head(15))

def listen_ref(ref_uid, max_seconds=30, show_reference=True):
    r=refs.set_index('ref_uid').loc[ref_uid]
    x,sr=sf.read(str(safe(r.path)),frames=int(max_seconds*r.sr),always_2d=True,dtype='float32')
    print('ref_uid:',ref_uid,'| duration:',r.duration,'| sr:',sr,'| channels:',x.shape[1])
    if show_reference and r.has_reference: print('Reference:',r.reference)
    display(Audio(x.T,rate=sr,normalize=False))
    t=np.arange(len(x))/sr
    fig,axs=plt.subplots(2,1,figsize=(12,5))
    for ch in range(x.shape[1]): axs[0].plot(t,x[:,ch],alpha=.65,label=f'channel {ch}')
    axs[0].set(xlabel='Seconds',ylabel='Amplitude'); axs[0].legend()
    if len(x)>256:
        f,tt,z=signal.stft(x.mean(axis=1),fs=sr,nperseg=min(len(x),round(.025*sr)),noverlap=None)
        axs[1].pcolormesh(tt,f,20*np.log10(np.abs(z)+1e-8),shading='auto',vmin=-90,vmax=-10)
        axs[1].set(xlabel='Seconds',ylabel='Hz',ylim=(0,min(8000,sr/2)))
    plt.tight_layout(); plt.show()

# Tự chọn ID từ queue. Audio chỉ phát trong notebook riêng tư.
# listen_ref(review.iloc[0].ref_uid)


## G. Benchmark tùy chọn — chọn model bằng kết quả, không bằng EDA
Ba checkpoint khởi điểm để thử, **không phải bảng xếp hạng LifeMate**:
- [openai/whisper-small](https://huggingface.co/openai/whisper-small): baseline Whisper đa ngôn ngữ.
- [vinai/PhoWhisper-small](https://huggingface.co/vinai/PhoWhisper-small): checkpoint PhoWhisper cho tiếng Việt.
- [nguyenvulebinh/wav2vec2-base-vietnamese-250h](https://huggingface.co/nguyenvulebinh/wav2vec2-base-vietnamese-250h): baseline CTC tiếng Việt.

Kiểm tra license, provenance, tài nguyên và điều kiện dùng pretrained/external data theo luật cuộc thi trước khi bật. Model cards được đối chiếu khi viết notebook; không tải trọng số hoặc gọi API bên ngoài ở chế độ EDA.

**Cách chạy:** chuẩn bị `transformers`, `torch`, `psutil`; nếu thiếu có thể cài riêng khi được phép. Cell không tự nâng phiên bản. Có thể thay `model_id` bằng thư mục checkpoint đã mount khi không có Internet. Lần đầu tải model có thể cần Internet; chỉ tải checkpoint, không gửi audio đi dịch vụ.

Harness dùng greedy thống nhất, batch size 1, không prompt/VAD/denoise, Whisper đặt language=vi/task=transcribe. Chỉ benchmark file **<=30 giây**, ghi rõ coverage; không âm thầm cắt file dài. Audio >30s cần benchmark riêng với chia đoạn/long-form được xác minh. Nếu có stereo nghi triệt tiêu, phải review/chốt chiến lược kênh trước; mặc định harness dùng mean channels.

Tập so sánh được cố định một lần trước khi chạy model, gồm final và history. Error decode/OOM được giữ trong kết quả và chấm như transcript rỗng (không xóa để làm đẹp WER). Nếu model không load được, báo `load_error`, không cho điểm giả. Không tune bằng public test.


In [ ]:
CANDIDATES=[
    {'name':'whisper_small','model_id':'openai/whisper-small','family':'seq2seq','revision':'main'},
    {'name':'phowhisper_small','model_id':'vinai/PhoWhisper-small','family':'seq2seq','revision':'main'},
    {'name':'wav2vec2_vi','model_id':'nguyenvulebinh/wav2vec2-base-vietnamese-250h','family':'ctc','revision':'main'},
]
# Chỉ sửa trước khi so sánh. Sau khi chốt: lưu revision commit cụ thể thay cho main.
bench_pool=validation[validation.duration.gt(0)&validation.duration.le(30)].copy()
bench_pool=bench_pool.drop_duplicates(['path','reference_norm'])
# Stratify round-robin theo final/history, độ dài, nhóm nội dung quan trọng.
bench_pool['bench_bin']=pd.cut(bench_pool.duration,[0,3,10,30],include_lowest=True).astype(str)
bench_pool['critical_hint']=bench_pool[['money_hint','time_hint','negation_hint','person_hint']].any(axis=1)
bs=[list(g.sample(frac=1,random_state=SEED).index) for _,g in
    bench_pool.groupby(['kind','bench_bin','critical_hint'],observed=True)]
chosen=[]
while any(bs) and len(chosen)<BASELINE_N:
    for b in bs:
        if b and len(chosen)<BASELINE_N: chosen.append(b.pop())
benchmark=bench_pool.loc[chosen].copy()
save_csv(benchmark,'benchmark_manifest.csv')
print('Validation refs:',len(validation),'| eligible <=30s unique pairs:',len(bench_pool),'| benchmark:',len(benchmark))
print('Không chấm các file >30s bằng harness này; cần bài đo long-form riêng.')
display(benchmark.groupby('kind').size().rename('benchmark_refs'))


In [ ]:
def resample_mono(path,target_sr):
    x,sr=sf.read(str(safe(path)),dtype='float32',always_2d=True)
    x=x.mean(axis=1)
    if sr!=target_sr:
        g=math.gcd(int(sr),int(target_sr))
        x=signal.resample_poly(x,target_sr//g,sr//g).astype('float32')
    return np.ascontiguousarray(x,dtype='float32')

benchmark_predictions=[]; model_runs=[]
if RUN_BASELINES:
    assert len(benchmark), 'Validation/benchmark rỗng; giải quyết split trước.'
    import torch, psutil, threading
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoModelForCTC
    device='cuda' if torch.cuda.is_available() else 'cpu'
    process=psutil.Process()
    def sync():
        if device=='cuda': torch.cuda.synchronize()
    for spec in CANDIDATES:
        print('Running:',spec['name'],'on',device)
        model=processor=None
        started=time.perf_counter()
        try:
            processor=AutoProcessor.from_pretrained(spec['model_id'],revision=spec['revision'],trust_remote_code=False)
            cls=AutoModelForSpeechSeq2Seq if spec['family']=='seq2seq' else AutoModelForCTC
            model=cls.from_pretrained(spec['model_id'],revision=spec['revision'],trust_remote_code=False)
            model=model.to(device).eval()
            target_sr=int(processor.feature_extractor.sampling_rate)
            load_seconds=time.perf_counter()-started
            resolved_revision=getattr(model.config,'_commit_hash',None)
            def forward_decode(x):
                batch=processor(x,sampling_rate=target_sr,return_tensors='pt')
                batch={k:v.to(device) for k,v in batch.items()}
                with torch.inference_mode():
                    if spec['family']=='seq2seq':
                        # Xóa forced_decoder_ids legacy; language/task được đặt tường minh.
                        if hasattr(model.generation_config,'forced_decoder_ids'): model.generation_config.forced_decoder_ids=None
                        ids=model.generate(**batch,language='vi',task='transcribe',num_beams=1,do_sample=False,max_new_tokens=440)
                        saturated=bool(ids.shape[-1]>=440)
                    else:
                        ids=model(**batch).logits.argmax(dim=-1); saturated=False
                return processor.batch_decode(ids)[0],saturated
            # Warm-up một input cố định; không đo vào steady-state latency.
            warm=resample_mono(benchmark.iloc[0].path,target_sr)
            forward_decode(warm); sync()
            for _,r in benchmark.iterrows():
                if device=='cuda': torch.cuda.reset_peak_memory_stats()
                peak=[process.memory_info().rss]; stop=threading.Event()
                def monitor():
                    while not stop.wait(.02): peak[0]=max(peak[0],process.memory_info().rss)
                watcher=threading.Thread(target=monitor,daemon=True); watcher.start()
                sync(); t0=time.perf_counter()
                status='ok'; error=''; hyp=''; saturated=False; compute_seconds=np.nan
                try:
                    x=resample_mono(r.path,target_sr)
                    sync(); t1=time.perf_counter()
                    hyp,saturated=forward_decode(x)
                    sync(); compute_seconds=time.perf_counter()-t1
                except Exception as e:
                    status='error'; error=f'{type(e).__name__}: {e}'[:300]
                    if device=='cuda': torch.cuda.empty_cache()
                finally:
                    sync(); e2e=time.perf_counter()-t0
                    peak[0]=max(peak[0],process.memory_info().rss); stop.set(); watcher.join()
                benchmark_predictions.append({'model':spec['name'],'ref_uid':r.ref_uid,'hypothesis':hyp,
                    'status':status,'error':error,'e2e_seconds':e2e,'compute_seconds':compute_seconds,
                    'rtf':e2e/r.duration,'rss_peak_sampled_mb':peak[0]/2**20,
                    'cuda_peak_allocated_mb':torch.cuda.max_memory_allocated()/2**20 if device=='cuda' else np.nan,
                    'generation_limit_flag':saturated})
            model_runs.append(spec|{'status':'ok','device':device,'target_sr':target_sr,'load_seconds':load_seconds,
                                    'resolved_revision':resolved_revision,'dtype':str(next(model.parameters()).dtype)})
        except Exception as e:
            model_runs.append(spec|{'status':'load_or_warmup_error','error':f'{type(e).__name__}: {e}'[:400]})
        finally:
            del model,processor
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        save_csv(pd.DataFrame(benchmark_predictions),'predictions.csv')
        (OUT/'model_runs.json').write_text(json.dumps(model_runs,ensure_ascii=False,indent=2),encoding='utf-8')
else:
    print('Benchmark tắt. Bật RUN_BASELINES sau khi kiểm tra schema, luật và validation.')


### WER/CER, failure rate và latency trên cùng tập
`WER=(S+D+I)/N`: thay thế, xóa, chèn chia cho số token tham chiếu. CER dùng ký tự. Notebook tính **corpus WER/CER** bằng tổng số lỗi / tổng độ dài reference, không lấy trung bình tỷ lệ từng câu. CER dưới đây bỏ khoảng trắng sau normalization. WER có thể >100% do chèn nhiều.

Hai chế độ: NFC+casefold+gộp khoảng trắng (giữ dấu câu), và thêm bỏ dấu câu. Không bỏ dấu tiếng Việt/negation, không chuẩn hóa số ngầm. Vì vậy “250000” và “hai trăm năm mươi nghìn” vẫn khác; nếu cần scoring normalization khác phải định nghĩa và áp dụng giống nhau cho reference/hypothesis.

`e2e_seconds`: đọc+downmix+resample+processor+model+decode; không gồm model loading và warm-up. `compute_seconds`: processor+model+decode, không chỉ riêng GPU kernels. RTF = e2e / thời lượng audio. RAM là **RSS toàn process lấy mẫu 20ms**, không phải phần riêng model và có thể bỏ lỡ spike; VRAM là peak allocated PyTorch, không phải toàn GPU. Số đo batch=1, một lần/file, không đại diện throughput batch hoặc độ ổn định qua nhiều lần chạy.


In [ ]:
def edit_counts(ref_tokens,hyp_tokens):
    # DP Levenshtein có đếm S,D,I; tie-break cố định substitution > deletion > insertion.
    previous=[(j,0,0,j) for j in range(len(hyp_tokens)+1)]
    for i,a in enumerate(ref_tokens,1):
        current=[(i,0,i,0)]
        for j,b in enumerate(hyp_tokens,1):
            if a==b: current.append(previous[j-1]); continue
            cost,s,d,ins=previous[j-1]; sub=(cost+1,s+1,d,ins)
            cost,s,d,ins=previous[j]; delete=(cost+1,s,d+1,ins)
            cost,s,d,ins=current[j-1]; insert=(cost+1,s,d,ins+1)
            current.append(min([sub,delete,insert],key=lambda z:z[0]))
        previous=current
    _,s,d,i=previous[-1]
    return s,d,i

def score_pair(ref,hyp,punctuation):
    r=normalize_text(ref,punctuation); h=normalize_text(hyp,punctuation)
    s,d,i=edit_counts(r.split(),h.split())
    cr=list(r.replace(' ','')); ch=list(h.replace(' ',''))
    cs,cd,ci=edit_counts(cr,ch)
    return {'S':s,'D':d,'I':i,'N':len(r.split()),'char_errors':cs+cd+ci,'char_N':len(cr),
            'exact_match':r==h}

predictions=pd.DataFrame(benchmark_predictions)
scored=pd.DataFrame(); leaderboard=pd.DataFrame(); slices=pd.DataFrame()
if len(predictions):
    scored=predictions.merge(benchmark,on='ref_uid',validate='many_to_one')
    metric_rows=[]
    for _,r in scored.iterrows():
        for mode,punct in [('keep_punctuation',False),('light_normalized',True)]:
            metric_rows.append(r.to_dict()|{'normalization':mode}|score_pair(r.reference,r.hypothesis,punct))
    scored=pd.DataFrame(metric_rows)
    def summarize(g):
        n=g.N.sum(); cn=g.char_N.sum()
        return pd.Series({'n':len(g),'WER':(g.S.sum()+g.D.sum()+g.I.sum())/n if n else np.nan,
            'CER':g.char_errors.sum()/cn if cn else np.nan,'exact_match':g.exact_match.mean(),
            'failure_rate':g.status.ne('ok').mean(),'latency_p50_s':g.e2e_seconds.median(),
            'latency_p95_s':g.e2e_seconds.quantile(.95),'RTF_total':g.e2e_seconds.sum()/g.duration.sum(),
            'max_sampled_RSS_MB':g.rss_peak_sampled_mb.max(),'max_VRAM_alloc_MB':g.cuda_peak_allocated_mb.max()})
    leaderboard=scored.groupby(['model','normalization']).apply(summarize)
    display(leaderboard)
    light=scored[scored.normalization=='light_normalized'].copy()
    slice_rows=[]
    for model,g in light.groupby('model'):
        for name in ['all','final','history']+list(patterns)+['flag_low_level','flag_near_fullscale']:
            sub=g if name=='all' else g[g.kind==name] if name in {'final','history'} else g[g[name].fillna(False)]
            if len(sub): slice_rows.append({'model':model,'slice':name}|summarize(sub).to_dict())
    slices=pd.DataFrame(slice_rows); display(slices)
    save_csv(scored,'scored_predictions.csv'); save_csv(leaderboard.reset_index(),'leaderboard.csv'); save_csv(slices,'slice_metrics.csv')
    print('Slice có entity-hint là WER trên câu có cờ; KHÔNG phải entity error rate.')
else:
    print('Chưa có predictions; phần EDA vẫn hoàn tất. Không tạo bảng xếp hạng giả.')


### Khoảng bất định khi so sánh hai model
Paired bootstrap theo `group_id` của validation, dùng cùng các sample cho hai model. Kết quả là khoảng percentile trên **benchmark đã chọn**, không tự suy rộng thành điểm thi hoặc toàn dataset. Nếu nhóm độc lập ít thì khoảng không đáng tin; notebook báo số nhóm. Giá trị delta WER âm nghĩa là model B tốt hơn model A trên phép đo này.


In [ ]:
def paired_bootstrap_wer(scored_df, model_a, model_b, repeats=1000):
    light=scored_df[scored_df.normalization=='light_normalized'].copy()
    light['errors']=light.S+light.D+light.I
    a=light[light.model==model_a].set_index('ref_uid')
    b=light[light.model==model_b].set_index('ref_uid')
    if set(a.index)!=set(b.index) or not len(a):
        raise ValueError('Hai model phải có predictions trên đúng cùng tập ref_uid, bao gồm failure.')
    joined=a[['group_id','errors','N']].rename(columns={'errors':'errors_a'}).join(b[['errors']].rename(columns={'errors':'errors_b'}))
    agg=joined.groupby('group_id')[['errors_a','errors_b','N']].sum().to_numpy()
    if len(agg)<2: return {'independent_groups':len(agg),'warning':'Không đủ nhóm để bootstrap'}
    local_rng=np.random.default_rng(SEED)
    deltas=[]
    for _ in range(repeats):
        z=agg[local_rng.integers(0,len(agg),len(agg))].sum(axis=0)
        deltas.append((z[1]-z[0])/z[2])
    return {'A':model_a,'B':model_b,'independent_groups':len(agg),
            'delta_WER_B_minus_A':float((agg[:,1].sum()-agg[:,0].sum())/agg[:,2].sum()),
            'percentile_95_low':float(np.quantile(deltas,.025)),
            'percentile_95_high':float(np.quantile(deltas,.975)),
            'warning':'Ít nhóm; diễn giải thận trọng' if len(agg)<30 else ''}

bootstrap_rows=[]
if len(scored):
    names=list(scored.model.unique())
    for other in names[1:]: bootstrap_rows.append(paired_bootstrap_wer(scored,names[0],other))
    display(pd.DataFrame(bootstrap_rows))
save_csv(pd.DataFrame(bootstrap_rows),'paired_bootstrap.csv')


### Chấm riêng thông tin có thể đổi hành động
WER trên câu có cờ chưa cho biết có sai chính entity không. Điền gold spans theo transcript **trước khi xem dự đoán**. Các cột money/time/location/device/person/negation là thông tin được nói, không suy từ environment. Ví dụ “sáng mai” giữ nguyên, không tự gán 08:00. Mỗi span có thể có các cách viết tương đương đã duyệt thủ công.

Sau đó điền phán đoán `correct/incorrect/uncertain/not_applicable` cho từng model. Đây là template đánh giá thủ công, không bịa bộ trích xuất entity hoặc gọi độ trùng regex là semantic accuracy. Phần tool calls/final state do nhóm agent đánh giá riêng qua interface hợp lệ; không mở state.json để chấm trong notebook ASR.


In [ ]:
gold=benchmark[['ref_uid','reference']].copy()
for c in ['person_spans_json','money_spans_json','time_spans_json','location_spans_json','device_spans_json','negation_spans_json']:
    gold[c]='[]'
save_csv(gold,'critical_gold_annotation.csv')
if len(predictions):
    judgments=predictions[['model','ref_uid','hypothesis','status']].merge(benchmark[['ref_uid','reference']],on='ref_uid')
else:
    judgments=pd.DataFrame(columns=['model','ref_uid','hypothesis','status','reference'])
for c in ['person_judgment','money_judgment','time_judgment','location_judgment','device_judgment','negation_judgment','notes']:
    judgments[c]=''
save_csv(judgments,'critical_model_review.csv')

def summarize_manual_judgments(path):
    df=pd.read_csv(path).fillna('')
    result=[]
    for model,g in df.groupby('model'):
        for c in [x for x in g if x.endswith('_judgment')]:
            counts=g[c].value_counts(); n=int(counts.get('correct',0)+counts.get('incorrect',0))
            result.append({'model':model,'entity':c,'adjudicated':n,
                'error_rate':counts.get('incorrect',0)/n if n else np.nan,
                'uncertain':int(counts.get('uncertain',0)),'unfilled':int(counts.get('',0))})
    return pd.DataFrame(result)


## H. Kết luận tự động có giới hạn và bàn giao
Notebook đưa ra **sự kiện đo được** và checklist quyết định. Nó không chọn model thắng chỉ vì audio 16 kHz, transcript tiếng Việt, hay một model có nhiều tham số.

Ưu tiên quyết định:
1. Đúng quy định + chạy được tài nguyên và giới hạn thời gian.
2. Tỷ lệ lỗi tên/số/thời gian/phủ định sau annotation có đủ mẫu.
3. WER/CER, failure rate và các nhóm khó trên cùng tập.
4. Latency/RTF/RAM/VRAM; long-form và history coverage.
5. Nếu chỉ hơn nhẹ trên tập nhỏ: tăng tập đánh giá, bootstrap theo sample hội thoại/nhóm độc lập trước khi kết luận. Không dùng từng token như quan sát độc lập.

Interface bàn giao gợi ý: `transcribe(audio_path) -> {text_model, text_normalized, model_id, revision, processing_config, latency_ms, status, warnings}`. Timestamps chỉ thêm nếu được hỗ trợ và kiểm tra. Không sinh confidence giả. Giữ audio ID để trace nhưng không gửi transcript chuẩn vào inference.


In [ ]:
facts={
 'audio_files':len(audio_df),'samples':len(samples),'references':len(refs),
 'resolved_references':int(refs.path.notna().sum()),'missing_or_ambiguous_references':int(refs.path.isna().sum()),
 'header_failures':int(audio_df.header_error.notna().sum()),
 'signal_audited_ok':int(audio_df.signal_status.eq('ok').sum()),
 'audio_hours_readable':float(audio_df.duration.sum()/3600),
 'duration_p95_seconds':float(audio_df.duration.quantile(.95)),
 'duration_over_30s_files':int(audio_df.duration.gt(30).sum()),
 'speaker_metadata_coverage_train_refs':speaker_coverage,
 'validation_refs':len(validation),'benchmark_refs':len(benchmark),
 'near_duplicate_scan_truncated':truncated,'parse_errors':len(parse_errors),
 'run_baselines':RUN_BASELINES,
}
recommendations=[]
if facts['missing_or_ambiguous_references'] or (stats.final_audio_refs==0).any():
    recommendations.append('P0: sửa/duyệt adapter và path resolution trước khi kết luận; chưa đủ coverage.')
if facts['parse_errors']: recommendations.append('P0: xử lý JSON parse errors; không coi manifest bị thiếu là đầy đủ.')
if facts['header_failures']: recommendations.append('P0: xem audio decode errors/codec; chưa được âm thầm bỏ file lỗi.')
if facts['duration_over_30s_files']: recommendations.append('Cần kiểm tra long-form/chia đoạn và benchmark riêng file >30s.')
if audio_df.channels.gt(1).any(): recommendations.append('Nghe kiểm tra stereo và channel cancellation trước khi chốt downmix.')
recommendations += [
 'Accent/noise/code-switching: cần nhãn nghe kiểm tra; heuristic văn bản không xác minh phân bố.',
 'Chạy các checkpoint ứng viên trên cùng benchmark; giữ failure và đo tài nguyên, không suy model tốt nhất từ EDA.',
 'Đánh giá riêng final/history; lỗi history có thể truyền sang hiểu yêu cầu cuối.',
 'Chấm critical entities thủ công và kiểm tra end-to-end agent độc lập.',
 'Không đề xuất fine-tune trước khi có phân tích lỗi baseline và kiểm tra quy định.',
]
report=['# LifeMate ASR audit — observed facts', '', '```json',json.dumps(facts,ensure_ascii=False,indent=2),'```',
        '', '## Next decisions']+['- '+x for x in recommendations]
report+=['','## Limits','- Audio stats are descriptive; no validated SNR/accent inference.',
         '- Exact hashes do not detect gain/resample/trimmed duplicates; near-text scan is bounded.',
         '- No test tuning; no environment/state access; no tool-call score in this notebook.',
         '- Model ranking requires actual benchmark outputs, critical annotation and resource constraints.']
(OUT/'audit_summary.md').write_text('\n'.join(report),encoding='utf-8')
config={'ROOT':str(ROOT),'SEED':SEED,'MAX_SIGNAL_FILES':MAX_SIGNAL_FILES,'MAX_SIGNAL_SECONDS':MAX_SIGNAL_SECONDS,
        'RUN_FILE_HASH':RUN_FILE_HASH,'VAL_FRACTION':VAL_FRACTION,'USE_SPEAKER_GROUPS':USE_SPEAKER_GROUPS,
        'TEXT_KEYS':list(TEXT_KEYS),'AUDIO_KEYS':sorted(AUDIO_KEYS),'SPLIT_OVERRIDES':SPLIT_OVERRIDES,
        'PATH_OVERRIDES':PATH_OVERRIDES,'RUN_BASELINES':RUN_BASELINES,'BASELINE_N':BASELINE_N,
        'CANDIDATES':CANDIDATES,'versions':versions}
(OUT/'run_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown('\n'.join(report)))
print('Outputs riêng tư:',OUT)
print('Không publish output cuộc thi. Không thể kết luận model tốt nhất nếu chưa có benchmark thực tế.')
